# Capítulo 9 — Reducir el número de dimensiones · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch09_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden: mientras falte una respuesta, las comprobaciones lo indican sin error. La solución está en `ch09_corrige.ipynb`, en la misma página complementaria.

**Convención.** Como en el libro, las medias, las varianzas y las covarianzas dividen por $n$.

**Índice.** Datos · Los cálculos de las lecciones 9.1 a 9.3 · Ejercicios 9.1 a 9.6

**Antes de abrir Jupyter:** los ejercicios de este capítulo se hacen primero en papel, con una calculadora. Las celdas de abajo sirven para **comprobar** tus resultados.

**Código de colores:** azul = viviendas; rombo negro = G; naranja = proyecciones; magenta discontinuo = lo que pierde la proyección.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Código de colores del libro: viviendas en azul, G como rombo negro, proyecciones en naranja,
# lo que pierde la proyección en magenta (discontinuo)
BLUE, ORANGE, MAGENTA, INK = '#2a78d6', '#eb6834', '#c2185b', '#0b0b0b'


def project(X, u):
    # coordenadas de los puntos (filas de X) sobre la dirección u, llevada a norma 1 (ecuación 9.1)
    u = np.asarray(u, dtype=float)
    return np.asarray(X, dtype=float) @ (u / np.linalg.norm(u))


def cov_matrix(X):
    # matriz de covarianza de las columnas de X, con una división por n (ecuación 9.2)
    X = np.asarray(X, dtype=float)
    D = X - X.mean(axis=0)
    return D.T @ D / len(X)

## Datos

El identificador D2.1 es el del libro y del archivo CSV. Los pequeños conjuntos de datos de los ejercicios se copian de los enunciados.

In [ ]:
# Datos del capítulo 9, copiados de los archivos CSV del libro (no modificar)

# D2.1 — las siete viviendas (A a F: ventas de referencia; G: la vivienda que hay que estimar)
# Columnas del CSV → nombres del notebook: logement → home, surface_m2 → floor_area (m²),
# distance_km → distance (km, distancia al centro), prix_kEUR → price (k€)
homes = pd.DataFrame([
    ('A', 32, 3.0, 118),
    ('B', 45, 1.5, 172),
    ('C', 54, 2.5, 185),
    ('D', 70, 1.0, 245),
    ('E', 85, 2.5, 279),
    ('F', 98, 0.5, 335),
    ('G', 62, 2.5, 209)
], columns=['home', 'floor_area', 'distance', 'price'])

reference = homes[homes['home'] != 'G']          # A a F: medias y desviaciones típicas de referencia
mu = reference[['floor_area', 'distance']].mean().to_numpy()
sigma = reference[['floor_area', 'distance']].std(ddof=0).to_numpy()   # división por n
Z = (reference[['floor_area', 'distance']].to_numpy() - mu) / sigma   # puntuaciones estandarizadas de A a F
z_G = (homes.loc[homes['home'] == 'G', ['floor_area', 'distance']].to_numpy()[0] - mu) / sigma
names = list(reference['home'])
print(homes.to_string(index=False))

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `home` | identificador de la vivienda (A, B…) | — | `logement` |
| `floor_area` | superficie habitable de la vivienda | m² | `surface_m2` |
| `price` | precio de venta | k€ | `prix_kEUR` |
| `distance` | distancia al centro de la ciudad | km | `distance_km` |
| `trainee` | identificador del alumno en prácticas | — | — |
| `math_score` | nota de matemáticas | puntos sobre 20 | — |
| `writing_score` | nota de redacción | puntos sobre 20 | — |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 9.1 a 9.3.

In [ ]:
# Lección 9.1 — proyectar sobre un eje (ecuación 9.1): eje de la superficie, después diagonal u = (1 ; −1)/√2
u_area = np.array([1.0, 0.0])
u_diag = np.array([1.0, -1.0]) / np.sqrt(2)
t = project(Z, u_diag)
t_G = project(z_G, u_diag)
print(pd.DataFrame({'home': names, 'z_area': Z[:, 0], 'z_distance': Z[:, 1], 't': t}).round(3).to_string(index=False))
print(f"G: t = {t_G:.3f}")
print(f"varianza conservada: eje de la superficie {np.mean(project(Z, u_area) ** 2):.3f}, diagonal {np.mean(t ** 2):.3f}")
print(f"varianza total {np.sum(Z.var(axis=0)):.3f} = conservada {np.mean(t ** 2):.3f} + perdida {np.sum(Z.var(axis=0)) - np.mean(t ** 2):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, u, title in zip(axes, (u_area, u_diag), ('eje de la superficie', 'diagonal')):
    shadows = np.outer(project(Z, u), u)
    for p, s in zip(Z, shadows):
        ax.plot([p[0], s[0]], [p[1], s[1]], color=MAGENTA, linestyle='--', linewidth=1)
    ax.axline((0, 0), u, color=ORANGE, linewidth=1)
    ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, zorder=3)
    ax.scatter(shadows[:, 0], shadows[:, 1], color=ORANGE, zorder=3)
    ax.scatter([z_G[0]], [z_G[1]], marker='D', color=INK, zorder=3)
    for n, p in zip(names + ['G'], list(Z) + [z_G]):
        ax.annotate(n, p, xytext=(4, 4), textcoords='offset points')
    ax.set_aspect('equal'); ax.set_title(title); ax.set_xlabel('z superficie'); ax.set_ylabel('z distancia')
plt.tight_layout(); plt.show()

In [ ]:
# Lección 9.2 — la matriz de covarianza (ecuación 9.2) y sus direcciones propias
products = Z[:, 0] * Z[:, 1]
print('productos z_s × z_d:', products.round(3), '→ suma', round(products.sum(), 3))
C = cov_matrix(Z)
print('matriz de covarianza C =\n', C.round(3))
for vec in ([1, 0], [1, -1], [1, 1]):
    print(f"C × {vec} = {(C @ np.array(vec, dtype=float)).round(3)}")
u1, u2 = u_diag
print(f"varianza sobre la diagonal, solo con C: {C[0, 0] * u1 ** 2 + 2 * C[0, 1] * u1 * u2 + C[1, 1] * u2 ** 2:.3f}")

In [ ]:
# Lección 9.3 — el ACP: valores propios ordenados de mayor a menor, proporción de varianza conservada (ecuación 9.3)
eigenvalues, eigenvectors = np.linalg.eigh(C)          # eigh: matriz simétrica, orden creciente
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
v1 = eigenvectors[:, 0] * np.sign(eigenvectors[0, 0])  # una dirección y su opuesta describen el mismo eje
print('valores propios:', eigenvalues.round(3), '→ suma', round(eigenvalues.sum(), 3))
print('primera dirección propia:', v1.round(3))
print('proporciones conservadas (%):', (100 * eigenvalues / eigenvalues.sum()).round(1))
pc1 = project(Z, v1)
print('ventas ordenadas por la primera componente:', [names[i] for i in np.argsort(pc1)],
      '; precios (k€):', list(reference['price'].to_numpy()[np.argsort(pc1)]))

# B y E: cercanos en la componente, alejados en el plano
q = project(Z, [1, 1])
iB, iE = names.index('B'), names.index('E')
print(f"B y E: diferencia en la componente {abs(pc1[iE] - pc1[iB]):.2f}, distancia en el plano "
      f"{np.linalg.norm(Z[iE] - Z[iB]):.2f}, diferencia en la segunda componente {abs(q[iE] - q[iB]):.2f}")

# Para ir más lejos: la SVD de Z da los mismos valores propios, s_k² / n
s = np.linalg.svd(Z, compute_uv=False)
print(f"valores singulares: {s.round(3)}; s² / n = {(s ** 2 / len(Z)).round(3)}")

## Ejercicio 9.1 — Proyectar sobre dos ejes

*Nociones: proyección y varianza conservada (lección 9.1, ecuación 9.1).* Tres puntos, $P_1 = (4 ; 2)$, $P_2 = (1 ; 3)$ y $P_3 = (-2 ; 1)$, y dos direcciones, $\mathbf{u} = (0{,}8 ; 0{,}6)$ y $\mathbf{v} = (-0{,}6 ; 0{,}8)$.

### a) a d) En papel

*Tus cálculos:* …

In [ ]:
# Los tres puntos (una fila por punto) y las dos direcciones
points = np.array([[4, 2], [1, 3], [-2, 1]], dtype=float)
u = np.array([0.8, 0.6])
v = np.array([-0.6, 0.8])

In [ ]:
# PARA COMPLETAR: coordenadas sobre u y sobre v, después sus varianzas (división por 3)
coords_u = None   # PARA COMPLETAR: points @ u
coords_v = None   # PARA COMPLETAR
var_u = None      # PARA COMPLETAR: np.var(coords_u) divide por n
var_v = None      # PARA COMPLETAR

In [ ]:
# Comprobación de a) a d)
if coords_u is None or coords_v is None or var_u is None or var_v is None:
    print('aún sin completar. Completa las cuatro líneas.')
else:
    print(f"a) normas {np.linalg.norm(u):.3f} y {np.linalg.norm(v):.3f}, producto escalar {u @ v:.3f}")
    print('b) sobre u:', np.round(coords_u, 2), '; sobre v:', np.round(coords_v, 2), '→',
          'correcto' if np.allclose(coords_u, [4.4, 2.6, -1.0]) and np.allclose(coords_v, [-0.8, 1.8, 2.0]) else 'revisar')
    print(f"c) P1: {coords_u[0] ** 2 + coords_v[0] ** 2:.2f} frente a una norma al cuadrado de {points[0] @ points[0]:.2f}")
    total = np.sum(points.var(axis=0))
    print(f"d) varianzas {var_u:.2f} y {var_v:.2f}, suma {var_u + var_v:.2f}, varianza total {total:.2f} →",
          'correcto' if abs(var_u - 5.04) < 1e-9 and abs(var_v - 1.6267) < 1e-3 and abs(var_u + var_v - total) < 1e-9 else 'revisar')

## Ejercicio 9.2 — Una matriz de covarianza a mano

*Nociones: covarianza (lección 1.5), matriz de covarianza (lección 9.2, ecuación 9.2).* Cuatro alumnos en prácticas han hecho dos pruebas, calificadas sobre 20 (conjunto de datos inventado).

- `trainee` — alumno en prácticas, S1 a S4;
- `math_score` — nota en la prueba de cálculo, sobre 20, en puntos;
- `writing_score` — nota en la prueba de redacción, sobre 20, en puntos.

In [ ]:
# Los cuatro alumnos en prácticas (ejercicios 9.2 a 9.5)
trainees = pd.DataFrame({
    'trainee':       ['S1', 'S2', 'S3', 'S4'],
    'math_score':    [15, 13, 9, 11],
    'writing_score': [11, 13, 9, 7],
})
scores = trainees[['math_score', 'writing_score']].to_numpy(dtype=float)
print(trainees.to_string(index=False))

### a) a c) En papel

*Tus cálculos:* …

In [ ]:
# PARA COMPLETAR: desviaciones respecto a la media, matriz de covarianza (división por 4) y correlación r
deviations = None   # PARA COMPLETAR: scores - scores.mean(axis=0)
M = None            # PARA COMPLETAR: deviations.T @ deviations / 4
r = None            # PARA COMPLETAR: M[0, 1] / raíz de M[0, 0] × M[1, 1]

In [ ]:
# Comprobación de a) a c)
if deviations is None or M is None or r is None:
    print('aún sin completar. Completa las tres líneas.')
else:
    print('a) medias:', scores.mean(axis=0), '; desviaciones:', deviations.tolist())
    print('b) y c) M =', np.asarray(M).tolist(), f"; r = {r:.2f} →",
          'correcto' if np.allclose(M, [[5, 3], [3, 5]]) and abs(r - 0.6) < 1e-9 else 'revisar')
    print('control con cov_matrix:', cov_matrix(scores).tolist())

## Ejercicio 9.3 — Encontrar las direcciones propias

*Nociones: dirección propia y valor propio (lección 9.2).* Retomamos la matriz del ejercicio 9.2, $M = \begin{pmatrix} 5 & 3 \\ 3 & 5 \end{pmatrix}$.

### a) a d) En papel

*Tus cálculos:* …

In [ ]:
# PARA COMPLETAR: valores propios de M, ordenados de mayor a menor, y proporción conservada por la mejor dirección
M93 = np.array([[5.0, 3.0], [3.0, 5.0]])
eigenvalues_93 = None   # PARA COMPLETAR: np.linalg.eigh(M93) devuelve (valores, vectores) en orden creciente
best_share = None       # PARA COMPLETAR: mayor valor propio / suma de los valores propios

In [ ]:
# Comprobación de a) a d)
if eigenvalues_93 is None or best_share is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    for vec in ([1, 0], [1, 1], [1, -1]):
        print(f"a) y b) M × {vec} = {(M93 @ np.array(vec, dtype=float)).tolist()}")
    print(f"valores propios {eigenvalues_93.round(3).tolist()}, proporción de la mejor dirección {100 * best_share:.0f} % →",
          'correcto' if np.allclose(eigenvalues_93, [8, 2]) and abs(best_share - 0.8) < 1e-9 else 'revisar')
    r_houses = -0.603
    print('d) r = −0,603: valores propios', np.sort(np.linalg.eigh(np.array([[1, r_houses], [r_houses, 1]]))[0])[::-1].round(3).tolist())

## Ejercicio 9.4 — Calcular una primera componente

*Nociones: ACP (lección 9.3), proyección (ecuación 9.1).* Retomamos las desviaciones respecto a la media del ejercicio 9.2. Aquí no se estandariza: las dos pruebas tienen la misma unidad y la misma varianza. La primera dirección propia es $\mathbf{v}_1 = (1 ; 1)/\sqrt{2}$.

### a) a d) En papel

*Tus cálculos y tu interpretación:* …

In [ ]:
# PARA COMPLETAR: coordenadas sobre v1, varianza, después reconstrucción de las notas de S1
v1_trainees = np.array([1.0, 1.0]) / np.sqrt(2)
t_trainees = None        # PARA COMPLETAR: desviaciones respecto a la media (ejercicio 9.2) @ v1_trainees
reconstructed_S1 = None  # PARA COMPLETAR: medias + t_trainees[0] × v1_trainees

In [ ]:
# Comprobación de a) a c)
if t_trainees is None or reconstructed_S1 is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    var_t = np.mean(t_trainees ** 2)
    error = scores[0] - reconstructed_S1
    print('a) coordenadas:', t_trainees.round(2).tolist(), f"; varianza {var_t:.2f}")
    print(f"b) proporción conservada {100 * var_t / np.sum(scores.var(axis=0)):.0f} %")
    print('c) notas reconstruidas de S1:', reconstructed_S1.round(2).tolist(), '; error:', error.round(2).tolist(),
          f"; longitud {np.linalg.norm(error):.2f} →",
          'correcto' if np.allclose(t_trainees, [2 * np.sqrt(2)] * 2 + [-2 * np.sqrt(2)] * 2)
          and np.allclose(reconstructed_S1, [14, 12]) else 'revisar')

## Ejercicio 9.5 — ¿Cercanos tras la proyección?

*Nociones: límites de la proyección (lección 9.3, ecuación 9.3).* Retomamos las coordenadas del ejercicio 9.4.

### a) a d) En papel

*Tus cálculos y tu demostración:* …

In [ ]:
# PARA COMPLETAR: distancias en el plano de las dos notas
distance_S1_S2 = None   # PARA COMPLETAR: np.linalg.norm(scores[0] - scores[1])
distance_S1_S3 = None   # PARA COMPLETAR

In [ ]:
# Comprobación de a), c) y d)
if distance_S1_S2 is None or distance_S1_S3 is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    t9 = (scores - scores.mean(axis=0)) @ (np.array([1.0, 1.0]) / np.sqrt(2))
    print(f"a) S1 y S2: diferencia proyectada {abs(t9[0] - t9[1]):.2f}, distancia {distance_S1_S2:.2f}")
    print(f"c) S1 y S3: diferencia proyectada {abs(t9[0] - t9[2]):.2f} ≤ distancia {distance_S1_S3:.2f} →",
          'correcto' if abs(distance_S1_S2 - np.sqrt(8)) < 1e-9 and abs(distance_S1_S3 - np.sqrt(40)) < 1e-9 else 'revisar')
    iB, iE = names.index('B'), names.index('E')
    t_BE = project(Z[[iB, iE]], [1, -1])
    q_BE = project(Z[[iB, iE]], [1, 1])
    print(f"d) B y E: diferencia en la componente {abs(t_BE[1] - t_BE[0]):.2f}, distancia {np.linalg.norm(Z[iE] - Z[iB]):.2f}, "
          f"parte perdida {abs(q_BE[1] - q_BE[0]):.2f}")

## Ejercicio 9.6 — Reto: el ACP sin estandarizar

*Nociones: matriz de covarianza (ecuación 9.2), ACP y unidades (lección 9.3).* Para las ventas A a F, trabajamos con las desviaciones brutas respecto a la media, sin estandarizar.

- `floor_area` — superficie de la vivienda, en m² (conjunto D2.1, columna `surface_m2`);
- `distance` — distancia al centro, en km (conjunto D2.1, columna `distance_km`).

### a) a d) En papel

*Tus cálculos y tu conclusión:* …

In [ ]:
# PARA COMPLETAR: matriz de covarianza bruta de A a F (superficie en m², distancia en km), después M × w
raw = reference[['floor_area', 'distance']].to_numpy(dtype=float)
M_raw = None   # PARA COMPLETAR: cov_matrix(raw)
w = np.array([1.0, -0.0238])
Mw = None      # PARA COMPLETAR: M_raw @ w

In [ ]:
# Comprobación de a) a d)
if M_raw is None or Mw is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    share = np.linalg.eigvalsh(M_raw)[-1] / np.trace(M_raw)
    raw_m = raw * [1, 1000]                       # distancia en metros
    share_m = np.linalg.eigvalsh(cov_matrix(raw_m))[-1] / np.trace(cov_matrix(raw_m))
    print('a) y b) M =', M_raw.round(3).tolist(), '; M × w =', Mw.round(2).tolist(), f"; cociente {Mw[1] / Mw[0]:.4f}")
    print(f"c) proporción conservada {100 * share:.1f} % (varianza total {np.trace(M_raw):.2f}) →",
          'correcto' if abs(M_raw[0, 1] + 74 / 6) < 1e-9 and abs(Mw[1] / Mw[0] + 0.0238) < 5e-5 else 'revisar')
    print(f"d) distancia en metros: varianza {cov_matrix(raw_m)[1, 1]:.0f} m², proporción conservada {100 * share_m:.2f} %")